# 🎬 استوديو الحي 404 — الحلقة الأولى «الشباك»
**مسار تجريبي مجاني من الموبايل.** كل الأكواد والسيناريو والأصوات الثلاثة والملفات الثابتة مجهّزة داخل GitHub. لا تحتاج لكتابة كود. شغّل الخلايا بزر **▶** بالترتيب.

**هتحتاج فقط:** حساب Google لتشغيل Colab، وحزمة `404_mobile_reference_assets.zip` التي أُرسلت في محادثة ChatGPT (حمّلها على الموبايل أولًا). الملف يجمع المراجع الأربعة الأصلية.

**مهم جدًا:** التشغيل هنا يُحضّر **أصواتًا مصرية تجريبية** ويحفظ كل ملفات المشاهد. هو **ليس** مولّد فيديو 3D احترافي جاهز، ولا ينتج حركة شفايف من تلقاء نفسه. فيديو 3D بمراجع ثابتة يحتاج توليد فيديو/ليب-سينك منفصل على GPU وقد يتجاوز حد Colab المجاني. عدم توفر GPU معناه التوقف، وليس فرض الدفع.

**ترخيص:** أصوات VoiceTut المدمجة للتقييم وليست مرخّصة تجاريًا حسب توضيح المطور. قبل النشر المربح يلزم استخدام أصوات أصلية مرخّصة.


In [ ]:
# الخطوة 1: جلب ملفات المشروع وفحص GPU — زر ▶
from pathlib import Path
import os,sys,subprocess,json
import torch
repo_dir=Path("/content/404")
if not (repo_dir/".git").exists():
    subprocess.run(["git","clone","--depth","1","https://github.com/omarnet0109-jpg/404.git",str(repo_dir)],check=True)
os.chdir(repo_dir)
subprocess.run([sys.executable,"production/check_episode.py"],check=True)
print("✅ السيناريو: 16 لقطة — 60 ثانية — 9 جمل مصرية — 3 شخصيات فقط")
print("GPU available:",torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:",torch.cuda.get_device_name(0))
else:
    print("⚠️ مش متاح GPU حاليًا. من Runtime > Change runtime type اختار GPU إن كانت متاحة مجانًا.")


In [ ]:
# الخطوة 2: رفع ملف المراجع الأصلي (ZIP واحد فقط) والتحقق من هوية الصور
from google.colab import files
import zipfile,hashlib
print("اختار 404_mobile_reference_assets.zip من Downloads على موبايلك.")
uploaded=files.upload()
archive_name="404_mobile_reference_assets.zip"
if archive_name not in uploaded:
    raise ValueError("اختار الملف المطلوب بالاسم 404_mobile_reference_assets.zip")
reference_dir=Path("/content/404/assets")
with zipfile.ZipFile(archive_name) as z:
    allowed={"references/ensemble.png","references/saber.png","references/layla.png","references/widad.png","references/manifest.json"}
    if set(z.namelist()) != allowed:
        raise ValueError("محتويات الأرشيف مش مطابقة لمراجع الشخصيات الأربع.")
    manifest=json.loads(z.read("references/manifest.json"))
    for ref in manifest["references"]:
        if hashlib.sha256(z.read(ref["path"])).hexdigest()!=ref["sha256"]:
            raise ValueError("صورة مرجعية تالفة أو اتغيرت: "+ref["path"])
    for member in z.namelist():
        target=reference_dir/member
        target.parent.mkdir(parents=True,exist_ok=True)
        target.write_bytes(z.read(member))
print("✅ تم تثبيت صور ليلى، الحاجة وداد، عم صابر والحارة بدون تعديل.")


In [ ]:
# الخطوة 3: معاينة أصوات الشخصيات المصرية الثلاثة (تحتاج GPU)
if not torch.cuda.is_available():
    raise RuntimeError("مفيش GPU مجانية متاحة في جلسة Colab دي. التجربة مش هتشتغل بشكل مضمون.")
subprocess.run([sys.executable,"-m","pip","install","-q","git+https://github.com/k2-fsa/OmniVoice.git"],check=True)
subprocess.run([sys.executable,"-m","pip","install","-q","voicetut-tts"],check=True)
from IPython.display import Audio,display
subprocess.run([sys.executable,"production/free_tts.py","--preview"],check=True)
for label,slug in [("ليلى","layla"),("الحاجة وداد","widad"),("عم صابر","saber")]:
    print("شغّل صوت:",label)
    display(Audio(filename=f"renders/episode-01/voice_previews/{slug}.wav"))
print("⚠️ قَيّم اللهجة والأعمار صوتيًا. دي أصوات تجريبية وليست أصوات نشر معتمدة.")


In [ ]:
#@title الخطوة 4: اعتماد أصوات الاختبار وتوليد الحوار التسع جُمل كما كُتبت
أوافق_على_الأصوات_التجريبية = False #@param {type:"boolean"}
if not أوافق_على_الأصوات_التجريبية:
    print("اسمع الصوت التجريبي لكل شخصية الأول، ثم فعّل المربع وشغّل الخلية تاني.")
else:
    import zipfile,json
    cfg=Path("config/free_voices.json")
    record=json.loads(cfg.read_text(encoding="utf-8"))
    record["voices_previewed_and_approved"]=True
    cfg.write_text(json.dumps(record,ensure_ascii=False,indent=2)+"\n",encoding="utf-8")
    subprocess.run([sys.executable,"production/free_tts.py","--episode"],check=True)
    voice_paths=sorted(Path("renders/episode-01/voices").glob("sh*.wav"))
    if len(voice_paths)!=9:
        raise RuntimeError(f"مطلوب 9 جمل، لكن اتولد {len(voice_paths)} فقط. مش هنبني فيديو ناقص.")
    archive=Path("/content/404_episode1_egyptian_test_voices.zip")
    with zipfile.ZipFile(archive,"w",compression=zipfile.ZIP_DEFLATED) as z:
        for path in voice_paths:
            z.write(path,arcname=path.name)
    print("✅ اتولد 9 ملفات WAV تجريبية")
    files.download(str(archive))


In [ ]:
# الخطوة 5: فحص جاهزية الفيديو النهائي — بدون ادعاء نجاح غير حقيقي
# ممنوع استبدال اللقطات الحوارية بمقاطع صامتة
import subprocess
result=subprocess.run([sys.executable,"production/assemble_episode.py","--check-only"],capture_output=True,text=True)
if result.returncode == 0:
    print("✅ كل مقاطع الفيديو والصوت موجودة؛ يجب مراجعة الشفايف والاستمرارية بصريًا.")
    print(result.stdout)
else:
    print("⏸️ الفيديو النهائي لم يُنتَج بعد؛ دي الملفات الناقصة بالضبط:")
    print(result.stderr[-5000:])


## حالة الاستوديو
- **جاهز في GitHub:** ملف الحلقة، مرجع الشخصيات، 16 لقطة، 9 جمل، فحص التوقيت، تركيب ملفات MP4 تلقائيًا.
- **يعمل من الموبايل إذا توفر GPU:** التجربة الصوتية المجانية ثم ملفات WAV.
- **غير جاهز للتوليد التلقائي هنا:** 16 لقطة 3D متحركة ومزامنة الشفايف. يلزم تشغيل نموذج فيديو متوافق على GPU ملائمة، وهذه القدرة ليست مضمونة في Colab المجاني.
- **عند غياب GPU:** لا يوجد اشتراك مطلوب، لكن لا يمكن ضمان استمرار الإنتاج بنفس المواصفات مجانًا.

المشروع الرسمي: [GitHub — omarnet0109-jpg/404](https://github.com/omarnet0109-jpg/404).
